In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ''
    for page in reader.pages:
        full_text += page.extract_text() + '\n'
    return full_text

# Usage
raw_text = extract_text_from_pdf(r"data/manual.pdf")

# Save to file
with open("raw.txt", "w", encoding="utf-8") as f:
    f.write(raw_text)


In [ ]:
import re

def clean_text(text):
    # Remove multiple newlines
    text = re.sub(r'\n+', '\n', text)
    # Remove unwanted characters
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # Remove non-ASCII
    text = re.sub(r'\s{2,}', ' ', text)         # Collapse multiple spaces
    return text.strip()

# Usage
with open("raw.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

cleaned_text = clean_text(raw_text)

with open("cleaned.txt", "w", encoding="utf-8") as f:
    f.write(cleaned_text)


In [ ]:
def chunk_text(text, min_words=200, max_words=500):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        end = i + max_words
        chunk = words[i:end]
        if len(chunk) >= min_words:
            chunks.append(" ".join(chunk))
        i += max_words
    return chunks

# Usage
chunks = chunk_text(cleaned_text)

# Save chunks to file (optional)
with open("chunks.txt", "w", encoding="utf-8") as f:
    for i, chunk in enumerate(chunks):
        f.write(f"--- Chunk {i+1} ---\n{chunk}\n\n")


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(chunks, show_progress_bar=True)

# Optional: Save embeddings
import numpy as np
np.save("embeddings.npy", embeddings)


In [ ]:
import faiss
import numpy as np

# Load embeddings and the chunks if not in memory
embeddings = np.load("embeddings.npy")  # if saved earlier

# Convert to float32 (required by FAISS)
embeddings = np.array(embeddings).astype('float32')

# Create FAISS index
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)  # L2 = Euclidean distance
index.add(embeddings)

# Save index (optional)
faiss.write_index(index, "faiss.index")


In [ ]:
def retrieve_top_chunks(query, model, index, chunks, top_k=5):
    query_embedding = model.encode([query])[0].astype('float32')
    distances, indices = index.search(np.array([query_embedding]), top_k)
    return [chunks[i] for i in indices[0]]

In [ ]:
import google.generativeai as genai

import os
from dotenv import load_dotenv
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

model = genai.GenerativeModel("gemini-2.0-flash")  #Make sure to use the correct model name according to the API documentation

def generate_with_gemini(question, context):
    prompt = f"""You are a helpful assistant with access to medical manual knowledge.

Answer the following question based on the context below.

Context:
{context}

Question:
{question}

Answer:"""

    try:
        response = model.generate_content(prompt)
        return response.text
    except Exception as e:
        print(f"Error generating response: {e}")
        return None


In [ ]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
query = "what is the difference between mental disorder and perosnality disorder"
top_chunks = retrieve_top_chunks(query, embed_model, index, chunks)
context = "\n\n".join(top_chunks)

answer = generate_with_gemini(query, context)
if answer:
    print(answer)
else:
    print("Failed to generate answer")
